# Estimación de Curvas Spot y Z-Spreads — Argentina
**Fecha de valuación:** 10 de abril de 2026  
**Metodología:** Nelson-Siegel-Svensson (NSS)  
**Familias analizadas:** Bonos Soberanos USD · CER (Tasa Real) · Dollar Linked

## Marco Teórico

### ¿Qué es una curva spot?
Una tasa spot s(t) es la tasa de interés que aplica hoy para un flujo de caja que ocurre exactamente en t años. A diferencia de la TIR —que promedia todos los flujos de un bono en una sola tasa— la curva spot asigna una tasa distinta a cada plazo. Esto permite valuar correctamente cualquier flujo futuro y calcular tasas forward implícitas.

### ¿Por qué Nelson-Siegel-Svensson en lugar de Bootstrap?
El bootstrap extrae tasas spot bono por bono de forma secuencial, produciendo una curva discreta que solo existe en los plazos donde hay bonos. NSS, en cambio, calibra una función continua y suave usando todos los bonos simultáneamente, minimizando el error cuadrático entre precios teóricos y de mercado. Esto lo hace más robusto ante precios con ruido y produce una curva evaluable en cualquier plazo.

### El modelo NSS
La tasa spot en función del plazo t está dada por:

$$s(t) = \beta_0 + \beta_1 \cdot \varphi(t,\tau_1) + \beta_2 \cdot [\varphi(t,\tau_1) - e^{-t/\tau_1}] + \beta_3 \cdot [\varphi(t,\tau_2) - e^{-t/\tau_2}]$$

donde $\varphi(t,\tau) = \frac{1 - e^{-t/\tau}}{t/\tau}$

Los 6 parámetros tienen interpretación económica directa:
- **β₀**: nivel de largo plazo — tasa a la que converge la curva cuando t → ∞
- **β₁**: pendiente — diferencia entre tasa corta y larga; negativo implica curva invertida
- **β₂, β₃**: curvaturas — controlan la presencia de "jorobas" en el tramo medio y largo
- **τ₁, τ₂**: escalas temporales — determinan en qué plazo aparecen las curvaturas

La calibración encuentra los 6 parámetros que minimizan $\sum_i (P_i^{mkt} - P_i^{NSS})^2$, donde cada precio teórico se calcula descontando los flujos del bono con la curva NSS: $P^{NSS} = \sum_t CF_t \cdot (1 + s(t))^{-t}$

### Z-Spread
El Z-Spread de una ON corporativa es el spread constante Z que sumado a toda la curva spot soberana iguala el precio de mercado de la ON:

$$P^{ON} = \sum_t \frac{CF_t}{(1 + s(t) + Z)^t}$$

Mide el rendimiento incremental que ofrece la ON por encima del soberano — es una medida del riesgo crediticio corporativo relativo al soberano del mismo tipo de moneda.

### Las tres familias de bonos argentinos
| Familia | Moneda de pago | Qué ajusta | Tasa que se estima |
|---------|---------------|-----------|-------------------|
| Hard Dollar | USD | Nada | Tasa en USD |
| CER | ARS | Inflación (IPC) | Tasa real en ARS |
| Dollar Linked | ARS | Tipo de cambio oficial | Tasa en USD implícita |

## Setup

In [ ]:
import sys
import warnings
import matplotlib
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

from main import main, CURVE_REGISTRY, SETTLEMENT_DATE, _run_curve_section, print_stats_table
from src import NSSCurve, SpreadEngine, Visualizer, data_loader

matplotlib.rcParams['figure.dpi'] = 120
print(f"Proyecto cargado correctamente. Settlement: {SETTLEMENT_DATE}")

---
## 1. Curva Soberana — Globales USD

Bonos utilizados: GD29, GD30, GD35, GD38, GD41, GD46  
Precio expresado como % del valor nominal original (base 100).  
Las ONs de referencia son YPF 2029 y Telecom 2031.

In [ ]:
main(curve="USD")

### Interpretación — USD
- La curva presenta forma **invertida**: las tasas cortas (~18%) superan ampliamente las largas (~8%), reflejando el riesgo de crédito soberano percibido en el corto plazo.
- **β₀ = 8.08%** representa el nivel estructural de largo plazo que el mercado le asigna a Argentina en dólares.
- **β₁ = 9.72%** confirma la fuerte pendiente negativa — típica de créditos con riesgo de refinanciamiento inminente.
- Los Z-Spreads positivos de YPF (+371 bps) y Telecom (+506 bps) reflejan el riesgo crediticio corporativo incremental sobre el soberano. La diferencia entre ambas ONs (135 bps) captura la prima de riesgo relativa entre las dos empresas.

---
## 2. Curva Soberana — CER (Tasa Real ARS)

Bonos utilizados: TX26, TX28, TX30, DICP, PARP  
Precio expresado como % del valor técnico (capital ajustado por CER).  
Las tasas estimadas son **tasas reales** — por encima de la inflación.  
Las ONs de referencia son Telecom Serie J (TLCJO) e IRSA (IRCPO).

In [ ]:
main(curve="CER")

### Interpretación — CER
- La curva real también es **invertida**: tasas reales cortas del ~10% contra ~4% en el largo plazo.
- **β₀ = 4.15%** es la tasa real estructural de largo plazo en pesos — elevada para estándares internacionales pero consistente con el historial inflacionario argentino.
- La inversión de la curva real puede reflejar expectativas de desinflación: si el mercado espera que la inflación baje, las tasas reales de corto plazo son altas hoy pero tenderán a comprimir.
- Z-Spreads positivos pero acotados (+60 y +77 bps) reflejan que el mercado CER corporativo es ilíquido y los spreads crediticios están comprimidos respecto al mercado USD.

---
## 3. Curva Soberana — Dollar Linked

Bonos utilizados: TV26D, TV27, TV28, TV30  
Precio expresado como % del valor técnico (capital ajustado por tipo de cambio oficial).  
Las tasas estimadas son **tasas en USD implícitas** — rendimiento en términos del TC oficial.  
Las ONs de referencia son Vista Oil (VISTAD28) y Pampa Energía (PAMPDL29).

In [ ]:
main(curve="DL")

### Interpretación — Dollar Linked
- La curva DL es la más plana de las tres: tasas cortas del ~8% convergiendo a ~2.7% de largo plazo.
- **β₀ = 1.80%** muy bajo refleja que el mercado no anticipa una devaluación sostenida del TC oficial en el largo plazo — consistente con el esquema de crawling peg vigente.
- Los Z-Spreads **negativos** de las ONs (-196 y -141 bps) no indican que las empresas sean menos riesgosas que el soberano. Reflejan la **escasez de instrumentos DL corporativos** en el mercado argentino: la alta demanda de cobertura cambiaria comprime los rendimientos de las ONs por debajo del soberano. Esta es una distorsión de mercado, no una señal crediticia.
- **Limitación metodológica:** con solo 4 bonos soberanos DL disponibles y 6 parámetros NSS a estimar, los grados de libertad son nulos y la calibración es exacta por construcción. Los resultados deben interpretarse con cautela.

---
## Conclusiones

### Comparación entre familias

| Familia | Tasa corta (~1y) | Tasa larga (β₀) | Forma | ONs Z-Spread |
|---------|-----------------|----------------|-------|-------------|
| Hard Dollar | ~18% | 8.08% | Invertida, pronunciada | +371 a +506 bps |
| CER (real) | ~10% | 4.15% | Invertida, moderada | +60 a +77 bps |
| Dollar Linked | ~8% | 1.80% | Invertida, suave | -196 a -141 bps* |

*Distorsión por iliquidez del mercado DL corporativo.

### Hallazgos principales
1. Las tres curvas soberanas presentan forma invertida, consistente con un crédito soberano percibido como riesgoso en el corto plazo pero con expectativas de normalización en el largo plazo.
2. El modelo NSS calibra con excelente ajuste en USD y CER (RMSE < $1). La familia DL tiene limitaciones por escasez de instrumentos.
3. Los Z-Spreads de las ONs hard dollar son elevados y diferenciados entre emisores, lo que confirma que el mercado discrimina riesgo crediticio corporativo incluso en un contexto de alto riesgo soberano.
4. La tasa real de largo plazo implícita en la curva CER (~4%) es consistente con las expectativas de estabilización macroeconómica pero sigue siendo elevada en términos históricos.